In [ ]:
pip install transformers accelerate datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Using cnn_dailymail as a reliable alternative for summarization
try:
    dataset = load_dataset("cnn_dailymail", "3.0.0", split='test')
    print("CNN/DailyMail dataset loaded successfully!")
    print(dataset)
    # Renaming for clarity in your context
    summarization_data = dataset
    display(pd.DataFrame(summarization_data.select(range(1))))
except Exception as e:
    print(f"Failed to load cnn_dailymail: {e}")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

CNN/DailyMail dataset loaded successfully!
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 11490
})


,article,highlights,id
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,f001ec5c4704938247d27a44948eebb37ae98d01


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Splitting the loaded dataset into a 80/20 train/test split
split_dataset = summarization_data.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

print(f"Training set size: {len(train_dataset)}")
print(f"Testing set size: {len(test_dataset)}")

# Displaying the structure of the split
print(split_dataset)

Training set size: 9192
Testing set size: 2298
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 9192
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2298
    })
})


In [ ]:
prefix = 'summarize:'

def process_function(examples):
  input = [prefix + doc for doc in examples['article']]
  model_input = tokenizer(input, max_length=1024, truncation=True)
  labels = tokenizer(examples['highlights'], max_length=128, truncation=True)
  model_input['labels'] = labels['input_ids']
  return model_input


In [ ]:
tokenized_dataset = split_dataset.map(process_function, batched = True)

Map:   0%|          | 0/9192 [00:00<?, ? examples/s]

Map:   0%|          | 0/2298 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model='t5-small')
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    weight_decay=.01,
    save_total_limit=3,
    num_train_epochs=10,
    fp16=True
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,2.010228,1.727707
2,1.887588,1.712488
3,1.872004,1.701116
4,1.833827,1.699938
5,1.842149,1.697866
6,1.838894,1.693648
7,1.815139,1.690925
8,1.813107,1.689985
9,1.807316,1.689445
10,1.799261,1.689601


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9200, training_loss=1.8483063175367271, metrics={'train_runtime': 6042.6153, 'train_samples_per_second': 15.212, 'train_steps_per_second': 1.523, 'total_flos': 2.486732406836429e+16, 'train_loss': 1.8483063175367271, 'epoch': 10.0})

In [ ]:
# Save the model and tokenizer
model_path = "./fine_tuned_t5_small"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model and tokenizer saved to {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./fine_tuned_t5_small


In [ ]:
import os

# Check if the folder exists
if not os.path.exists('./fine_tuned_t5_small'):
    print('Directory missing. Re-saving now...')
    try:
        trainer.save_model('./fine_tuned_t5_small')
        tokenizer.save_pretrained('./fine_tuned_t5_small')
        print('Model re-saved successfully.')
    except NameError:
        print('Error: The "trainer" object is gone. You must re-run the training cell (VUoTN5EfaY7i) first.')
else:
    print('Directory found:', os.listdir('./fine_tuned_t5_small'))

Directory found: ['model.safetensors', 'config.json', 'tokenizer.json', 'generation_config.json', 'training_args.bin', 'tokenizer_config.json']


### Inference with the Fine-Tuned Model
Now we can load the model and tokenizer from the saved path to generate summaries for new text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import evaluate
import os

model_path = "./fine_tuned_t5_small"

# 1. Ensure the model exists in the current runtime
if not os.path.exists(model_path):
    print("Model directory not found.")
    try:
        print("Attempting to save model from active trainer...")
        trainer.save_model(model_path)
        tokenizer.save_pretrained(model_path)
        print("Model saved successfully!")
    except NameError:
        print("Error: The 'trainer' object is not available in this session.")
        print("Please re-run the training cell (VUoTN5EfaY7i) to recreate the model.")
        raise FileNotFoundError("Model files missing and trainer not found.")

# 2. Load the model now that we've verified it exists
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 3. Generate summary and evaluate
sample_text = prefix + test_dataset[0]['article']
ground_truth = test_dataset[0]['highlights']

inputs = tokenizer(sample_text, return_tensors="pt", max_length=1024, truncation=True).to(device)
outputs = model.generate(inputs["input_ids"], max_length=128, min_length=30, do_sample=False)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Model Generated Summary ---")
print(summary)

rouge = evaluate.load('rouge')
results = rouge.compute(predictions=[summary], references=[ground_truth])
print("\n--- ROUGE Scores ---")
print(results)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


--- Model Generated Summary ---
a revolution in the attitudes of everyday Americans is a revolution, says adam sutter. sutter: weed is legal in your state, but it can bring you to tears.

--- ROUGE Scores ---
{'rouge1': np.float64(0.10526315789473684), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.07017543859649124), 'rougeLsum': np.float64(0.07017543859649124)}


### Download Model for Deployment
Run the cell below to compress the model directory into a `.zip` file and download it to your local machine.

In [ ]:
import shutil
from google.colab import files

# Name of the zip file
zip_filename = "fine_tuned_t5_small.zip"

# Compress the folder
shutil.make_archive("fine_tuned_t5_small", 'zip', "./fine_tuned_t5_small")

# Download the file
files.download(zip_filename)
print(f"Downloading {zip_filename}...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install evaluate

In [ ]:
!pip install rouge_score

In [ ]:
# This cell is no longer needed as ROUGE calculation has been moved to cell c84ea6bf
# import evaluate

# rouge = evaluate.load('rouge')

# # Compute ROUGE score
# results = rouge.compute(predictions=[summary], references=[ground_truth])
# print(results)